# **Evaluation**

In [24]:
import os
import sys

sys.path.append("../../")

from datetime import datetime, timedelta
from random import randint, random

from dotenv import load_dotenv
from metrics import (
    get_budget_efficiency,
    get_dietary_compliance,
    get_expiry_weighted_utilisation_score,
    get_ingredient_utilisation_score,
)
from openai import OpenAI

from engines import (
    GAMealPlanner,
    ILPMealPlanner,
    LLMMealPlanner,
    RandomMealPlanner,
    get_pantry_ingredient,
    load_all_ingredients,
    make_preferences,
)
from models import MealPlanningEnvironment, Pantry

load_dotenv()

True

## Evaluation Environment Setup

In [25]:
user_preferences = make_preferences()

In [26]:
all_ingredients = load_all_ingredients(filepath="../../recipe_extraction/supplemented_structured_ingredients.json")

In [27]:
pantry_ingredient_name_by_quantity = {
    "chicken breast": 800,
    "broccoli": 1500,
    "rice": 1000,
}

In [28]:
CURRENT_DATE = datetime.now()

In [29]:
pantry_ingredients = [
    get_pantry_ingredient(name, CURRENT_DATE + timedelta(days=randint(1, 7)), random(), all_ingredients)
    for name in pantry_ingredient_name_by_quantity.keys()
]

In [30]:
ingredient_costs = {}

for ingredient in pantry_ingredients:
    ingredient_costs[ingredient.name] = ingredient.estimated_financial_cost

In [31]:
pantry_ingredient_by_quantity = dict(zip(pantry_ingredients, pantry_ingredient_name_by_quantity.values()))

In [32]:
pantry = Pantry()

for pantry_ingredient, quantity in pantry_ingredient_by_quantity.items():
    pantry.add(
        pantry_ingredient,
        quantity,
    )

In [33]:
meal_planning_environment = MealPlanningEnvironment(
    recipes=[],
    pantry=pantry,
    preferences=user_preferences,
)

In [34]:
meal_planning_environment.load_recipes_from_json(
    filepath="../../recipe_extraction/supplemented_structured_recipes.json"
)

In [35]:
planner_metrics: dict[str, dict[str, float]] = {}

## Random Meal Planner

In [36]:
RANDOM_MEAL_PLANNER_SEED = 1

In [37]:
random_meal_planner = RandomMealPlanner(
    meal_planning_environment=meal_planning_environment, random_seed=RANDOM_MEAL_PLANNER_SEED
)

In [38]:
best_random_meal_plan, _ = random_meal_planner.generate_meal_plan()

In [39]:
best_random_meal_plan_recipes = [
    [meal_planning_environment.recipes[int(index)] for index in best_random_meal_plan[i : i + 3]]
    for i in range(0, len(best_random_meal_plan), 3)
]

In [40]:
random_dietary_compliance = get_dietary_compliance(
    meal_plan=best_random_meal_plan_recipes, user_preferences=user_preferences
)

In [41]:
random_ingredient_utilisation = get_ingredient_utilisation_score(meal_plan=best_random_meal_plan_recipes, pantry=pantry)

In [42]:
random_expiry_weighted_utilisation = get_expiry_weighted_utilisation_score(
    meal_plan=best_random_meal_plan_recipes, pantry=pantry, current_date=CURRENT_DATE
)

In [43]:
random_budget_efficiency = get_budget_efficiency(
    meal_plan=best_random_meal_plan_recipes,
    pantry=pantry,
    ingredient_costs=ingredient_costs,
    weekly_budget=user_preferences.weekly_budget,
)

In [44]:
planner_metrics[random_meal_planner.__class__.__name__] = {
    "dietary_compliance": random_dietary_compliance,
    "ingredient_utilisation": random_ingredient_utilisation,
    "expiry_weighted_utilisation": random_expiry_weighted_utilisation,
    "budget_efficiency": random_budget_efficiency,
}

## GA Meal Planner

In [12]:
ga_meal_planner = GAMealPlanner(meal_planning_environment=meal_planning_environment)

# TODO add params

## ILP Meal Planner

In [13]:
ilp_meal_planner = ILPMealPlanner(meal_planning_environment=meal_planning_environment)

# TODO add params

## LLM Meal Planner

In [ ]:
# TODO the LLM meal planner should have less recipes for context window

In [46]:
import random as rng

LLM_MAX_RECIPES = 10

llm_recipes = rng.sample(
    meal_planning_environment.recipes, min(LLM_MAX_RECIPES, len(meal_planning_environment.recipes))
)

llm_meal_planning_environment = MealPlanningEnvironment(
    recipes=llm_recipes,
    pantry=pantry,
    preferences=user_preferences,
)

print(f"LLM planner will use {len(llm_meal_planning_environment.recipes)} recipes")

LLM planner will use 10 recipes


In [47]:
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

llm_meal_planner = LLMMealPlanner(
    meal_planning_environment=llm_meal_planning_environment,
    llm_client=client,
    prompt_filepath="../LLMMealPlannerPrompt.txt",
)

In [48]:
best_llm_meal_plan, _ = llm_meal_planner.generate_meal_plan()

In [49]:
best_llm_meal_plan_recipes = [
    [llm_meal_planning_environment.recipes[int(index)] for index in best_llm_meal_plan[i : i + 3]]
    for i in range(0, len(best_llm_meal_plan), 3)
]

In [50]:
llm_dietary_compliance = get_dietary_compliance(meal_plan=best_llm_meal_plan_recipes, user_preferences=user_preferences)

In [51]:
llm_ingredient_utilisation = get_ingredient_utilisation_score(meal_plan=best_llm_meal_plan_recipes, pantry=pantry)

In [52]:
llm_expiry_weighted_utilisation = get_expiry_weighted_utilisation_score(
    meal_plan=best_llm_meal_plan_recipes, pantry=pantry, current_date=CURRENT_DATE
)

In [53]:
llm_budget_efficiency = get_budget_efficiency(
    meal_plan=best_llm_meal_plan_recipes,
    pantry=pantry,
    ingredient_costs=ingredient_costs,
    weekly_budget=user_preferences.weekly_budget,
)

In [54]:
planner_metrics[llm_meal_planner.__class__.__name__] = {
    "dietary_compliance": llm_dietary_compliance,
    "ingredient_utilisation": llm_ingredient_utilisation,
    "expiry_weighted_utilisation": llm_expiry_weighted_utilisation,
    "budget_efficiency": llm_budget_efficiency,
}

In [55]:
print(planner_metrics)

{'RandomMealPlanner': {'dietary_compliance': 0.0, 'ingredient_utilisation': 0.0, 'expiry_weighted_utilisation': 0.0, 'budget_efficiency': 0.2816676863869441}, 'LLMMealPlanner': {'dietary_compliance': 0.4816055842534013, 'ingredient_utilisation': 0.0, 'expiry_weighted_utilisation': 0.0, 'budget_efficiency': 0.49828117846185416}}


## Evaluation